# 完整建模流程演示 (Complete Workflow)

本notebook演示hscredit库完整的风控建模流程，从数据探索到模型训练的完整Pipeline。

In [ ]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n目标分布:")
print(df['FPD15'].value_counts())

## Step 1: 数据准备

In [ ]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"特征数量: {len(feature_cols)}")

# 划分训练集和测试集
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"训练集: {X_train.shape}, 坏样本率: {y_train.mean()*100:.2f}%")
print(f"测试集: {X_test.shape}, 坏样本率: {y_test.mean()*100:.2f}%")

## Step 2: 特征分箱

In [ ]:
from hscredit.core.binning import OptimalBinning

# 对所有特征进行分箱
n_bins = 5
binning_results = {}

for col in feature_cols[:10]:  # 使用前10个特征演示
    binner = OptimalBinning(method='quantile', max_n_bins=n_bins)
    binner.fit(X_train[[col]], y_train)
    binning_results[col] = binner

print(f"完成 {len(binning_results)} 个特征的分箱")

## Step 3: WOE编码

In [ ]:
from hscredit.core.encoders import WOEEncoder

# 使用分箱结果进行WOE编码
woe_encoder = WOEEncoder()
X_train_woe = woe_encoder.fit_transform(X_train[list(binning_results.keys())], y_train)
X_test_woe = woe_encoder.transform(X_test[list(binning_results.keys())])

print(f"WOE编码后形状: {X_train_woe.shape}")
print(f"\nWOE编码结果（部分）:")
print(X_train_woe.head())

## Step 4: 特征筛选

In [ ]:
from hscredit.core.selectors import IVSelector, CorrSelector

# IV筛选
iv_selector = IVSelector(threshold=0.02)
X_train_iv = iv_selector.fit_transform(X_train_woe, y_train)
X_test_iv = iv_selector.transform(X_test_woe)

print(f"IV筛选前特征数: {X_train_woe.shape[1]}")
print(f"IV筛选后特征数: {X_train_iv.shape[1]}")

# 相关性筛选
corr_selector = CorrSelector(threshold=0.8)
X_train_final = corr_selector.fit_transform(X_train_iv, y_train)
X_test_final = corr_selector.transform(X_test_iv)

print(f"相关性筛选后特征数: {X_train_final.shape[1]}")

## Step 5: 模型训练

In [ ]:
from hscredit.core.models import LogisticRegression
from hscredit.core.metrics import KS, AUC, Gini

# 逻辑回归模型
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_final, y_train)

# 预测
y_train_proba = lr_model.predict_proba(X_train_final)[:, 1]
y_test_proba = lr_model.predict_proba(X_test_final)[:, 1]

# 评估
train_ks = KS(y_train, y_train_proba)
train_auc = AUC(y_train, y_train_proba)
train_gini = Gini(y_train, y_train_proba)

test_ks = KS(y_test, y_test_proba)
test_auc = AUC(y_test, y_test_proba)
test_gini = Gini(y_test, y_test_proba)

print("模型性能:")
print(f"{'指标':<10} {'训练集':<10} {'测试集':<10}")
print(f"{'KS':<10} {train_ks:<10.4f} {test_ks:<10.4f}")
print(f"{'AUC':<10} {train_auc:<10.4f} {test_auc:<10.4f}")
print(f"{'Gini':<10} {train_gini:<10.4f} {test_gini:<10.4f}")

## Step 6: 生成评分

In [ ]:
from hscredit.core.models import ScoreCard

# 创建评分卡
scorecard = ScoreCard(
    binner=woe_encoder,
    coef=lr_model.coef_[0],
    intercept=lr_model.intercept_[0],
    pdo=50,
    rate=0.5,
    score=600,
    floor=300,
    ceiling=800
)

# 生成评分
train_scores = scorecard.predict(X_train_final)
test_scores = scorecard.predict(X_test_final)

print("评分统计:")
print(f"{'数据集':<10} {'最小值':<10} {'最大值':<10} {'均值':<10} {'标准差':<10}")
print(f"{'训练集':<10} {train_scores.min():<10.2f} {train_scores.max():<10.2f} {train_scores.mean():<10.2f} {train_scores.std():<10.2f}")
print(f"{'测试集':<10} {test_scores.min():<10.2f} {test_scores.max():<10.2f} {test_scores.mean():<10.2f} {test_scores.std():<10.2f}")

## Step 7: 分箱分段统计

In [ ]:
# 评分分段
def score_binning(scores, n_bins=10):
    bins = np.linspace(scores.min(), scores.max(), n_bins + 1)
    return np.digitize(scores, bins[:-1])

train_bins = score_binning(train_scores)
test_bins = score_binning(test_scores)

# 统计每个分段的样本数和坏账率
score_stats = pd.DataFrame({
    '分段': range(1, 11),
    '训练样本数': [np.sum(train_bins == i) for i in range(1, 11)],
    '训练坏账率': [y_train[train_bins == i].mean() for i in range(1, 11)],
    '测试样本数': [np.sum(test_bins == i) for i in range(1, 11)],
    '测试坏账率': [y_test[test_bins == i].mean() for i in range(1, 11)]
})

print("评分分段统计:")
print(score_stats.to_string(index=False))

## Step 8: 保存完整流程结果

In [ ]:
from hscredit.utils import save_pickle

# 创建输出目录
output_dir = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/workflow_results'
os.makedirs(output_dir, exist_ok=True)

# 保存模型
save_pickle(lr_model, os.path.join(output_dir, 'lr_model.pkl'))
save_pickle(woe_encoder, os.path.join(output_dir, 'woe_encoder.pkl'))
save_pickle(scorecard, os.path.join(output_dir, 'scorecard.pkl'))

# 保存结果到Excel
output_path = os.path.join(output_dir, 'workflow_results.xlsx')
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    pd.DataFrame({
        '指标': ['KS', 'AUC', 'Gini'],
        '训练集': [train_ks, train_auc, train_gini],
        '测试集': [test_ks, test_auc, test_gini]
    }).to_excel(writer, sheet_name='模型性能', index=False)

    score_stats.to_excel(writer, sheet_name='评分分段', index=False)

    pd.DataFrame({
        '样本ID': range(len(train_scores)),
        '训练评分': train_scores
    }).to_excel(writer, sheet_name='训练评分', index=False)

    pd.DataFrame({
        '样本ID': range(len(test_scores)),
        '测试评分': test_scores
    }).to_excel(writer, sheet_name='测试评分', index=False)

print(f"完整流程结果已保存至: {output_dir}")